# Exercise 1 — Sentiment Analysis with Hugging Face

**Practice 3 — Get Started with Hugging Face**  

Mục tiêu:
1. Cài thư viện `transformers`
2. Dùng pre-trained model từ Hugging Face Hub
3. Tokenize câu mẫu
4. Thực hiện sentiment analysis

## Step 1 — Install Hugging Face `transformers`

In [1]:
# Step 1: Install the Hugging Face transformers library
%pip install transformers torch --quiet
print("Installation complete.")

Note: you may need to restart the kernel to use updated packages.
Installation complete.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports    

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print(f"PyTorch  : {torch.__version__}")

import transformers
print(f"Transformers: {transformers.__version__}")


c:\Users\tuanmit399\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch  : 2.5.1+cu121
Transformers: 4.57.6


In [3]:
# ── Value Object ────────────────────────────────────────────────────
@dataclass(frozen=True)
class SentimentResult:
    """Immutable structured result from one inference call."""
    sentence:      str
    sentiment:     str           # "POSITIVE" or "NEGATIVE"
    confidence:    float         # probability of predicted class ∈ [0, 1]
    tokens:        list[str]     # decoded subword tokens
    probabilities: dict[str, float]


# ── Main Analyzer Class ─────────────────────────────────────────────
class SentimentAnalyzer:
    """Load a Hugging Face sentiment model and run inference.
    
    Implements Exercise 1:
      Step 2 — Load pre-trained model from Hub (via __init__)
      Step 3 — Tokenize a sentence       (via tokenize)
      Step 4 — Perform sentiment analysis (via analyze)
    """

    EXPECTED_LABELS = frozenset({"NEGATIVE", "POSITIVE"})

    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english") -> None:
        self.model_name = model_name
        # Step 2: Load tokenizer and model from Hugging Face Hub
        self._tokenizer = AutoTokenizer.from_pretrained(model_name)
        self._model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self._model.eval()   # inference mode — disable dropout
        self._validate_labels()
        print(f"Loaded model : {model_name}")

    def tokenize(self, sentence: str) -> dict[str, Any]:
        """Step 3: Tokenize sentence → input_ids + attention_mask."""
        sentence = sentence.strip()
        return self._tokenizer(sentence, return_tensors="pt")

    def analyze(self, sentence: str) -> SentimentResult:
        """Step 4: Classify sentence without updating weights."""
        sentence = sentence.strip()
        inputs   = self._tokenizer(sentence, return_tensors="pt")

        with torch.no_grad():          # no gradient computation
            logits = self._model(**inputs).logits

        probs        = torch.softmax(logits, dim=-1)[0]
        predicted_id = int(torch.argmax(probs))
        label_map    = self._model.config.id2label
        probabilities = {label_map[i]: float(p) for i, p in enumerate(probs)}
        sentiment     = label_map[predicted_id]
        tokens        = self._tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        return SentimentResult(
            sentence=sentence,
            sentiment=sentiment,
            confidence=probabilities[sentiment],
            tokens=tokens,
            probabilities=probabilities,
        )

    def _validate_labels(self) -> None:
        actual = set(self._model.config.id2label.values())
        if actual != self.EXPECTED_LABELS:
            raise ValueError(f"Expected {self.EXPECTED_LABELS}, got {actual}")

print("Classes defined: SentimentResult, SentimentAnalyzer")

Classes defined: SentimentResult, SentimentAnalyzer


## Step 2 — Load Pre-trained Model from Hugging Face Hub

In [4]:
# Step 2: Instantiate analyzer — downloads model on first run
MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
analyzer = SentimentAnalyzer(MODEL_ID)

# Show model architecture summary
total_params = sum(p.numel() for p in analyzer._model.parameters())
print(f"Total parameters : {total_params:,}")
print(f"Output labels    : {list(analyzer._model.config.id2label.values())}")

Loaded model : distilbert-base-uncased-finetuned-sst-2-english
Total parameters : 66,955,010
Output labels    : ['NEGATIVE', 'POSITIVE']


## Step 3 — Tokenize a Sample Sentence

In [5]:
# Step 3: Tokenize the demo sentence
SENTENCE = "I really enjoyed learning how to use Hugging Face Transformers."

encoded = analyzer.tokenize(SENTENCE)
tokens  = analyzer._tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

print(f"Sentence      : {SENTENCE}")
print(f"Tokens        : {tokens}")
print(f"Token IDs     : {encoded['input_ids'].tolist()}")
print(f"Attention mask: {encoded['attention_mask'].tolist()}")
print(f"Sequence len  : {encoded['input_ids'].shape[1]} tokens")

Sentence      : I really enjoyed learning how to use Hugging Face Transformers.
Tokens        : ['[CLS]', 'i', 'really', 'enjoyed', 'learning', 'how', 'to', 'use', 'hugging', 'face', 'transformers', '.', '[SEP]']
Token IDs     : [[101, 1045, 2428, 5632, 4083, 2129, 2000, 2224, 17662, 2227, 19081, 1012, 102]]
Attention mask: [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
Sequence len  : 13 tokens


## Step 4 — Perform Sentiment Analysis

In [6]:
# Step 4: Run sentiment analysis
result = analyzer.analyze(SENTENCE)

print("=" * 50)
print(f"  Sentence   : {result.sentence}")
print(f"  Sentiment  : {result.sentiment}")
print(f"  Confidence : {result.confidence:.2%}")
print("  Probabilities:")
for label, prob in result.probabilities.items():
    bar = "█" * int(prob * 30)
    print(f"    {label:10s}: {prob:.2%}  {bar}")
print("=" * 50)

  Sentence   : I really enjoyed learning how to use Hugging Face Transformers.
  Sentiment  : POSITIVE
  Confidence : 99.92%
  Probabilities:
    NEGATIVE  : 0.08%  
    POSITIVE  : 99.92%  █████████████████████████████


## Validation Checks

In [7]:
# Validation assertions
cls_token = analyzer._tokenizer.cls_token
sep_token = analyzer._tokenizer.sep_token

assert result.sentiment in {"POSITIVE", "NEGATIVE"}, "Invalid label"
assert 0.0 <= result.confidence <= 1.0, "Confidence out of range"
assert abs(sum(result.probabilities.values()) - 1.0) < 1e-5, "Probabilities don't sum to 1"
assert result.tokens[0]  == cls_token, f"Expected CLS token, got {result.tokens[0]}"
assert result.tokens[-1] == sep_token, f"Expected SEP token, got {result.tokens[-1]}"

print("All Exercise 1 validation checks passed. ✔")

All Exercise 1 validation checks passed. ✔


## Test with Other Sentences

Thử nghiệm thêm với các câu khác nhau:

In [8]:
test_sentences = [
    "This movie was absolutely terrible and boring.",
    "An outstanding performance by all the actors!",
    "It was okay, nothing special.",
]

print(f"{'Sentence':<50} {'Label':<10} {'Confidence'}")
print("-" * 75)
for s in test_sentences:
    r = analyzer.analyze(s)
    print(f"{s:<50} {r.sentiment:<10} {r.confidence:.2%}")

Sentence                                           Label      Confidence
---------------------------------------------------------------------------
This movie was absolutely terrible and boring.     NEGATIVE   99.98%
An outstanding performance by all the actors!      POSITIVE   99.99%
It was okay, nothing special.                      NEGATIVE   98.21%
